# CME538 - Introduction to Data Science
## Lecture 2.2 - Pandas II

### Goals

In this lecture, we continue working with **Pandas**. We will focus on two ideas that appear constantly in data analysis: working with **text data** and summarizing data using **groups**.

By the end of the lecture, you should be able to:

- use Pandas `.str` methods to work with text in a Series,
- create Boolean conditions from string operations,
- explain the **split → apply → combine** idea behind `.groupby()`,
- aggregate grouped data using built-in and custom functions, and
- distinguish between **row filtering** and **group filtering**.

As in Pandas I, the goal is not to memorize every method. Focus on the patterns and on reading Pandas expressions one step at a time.

### Lecture Structure

1. String `.str` Methods
2. Grouping with `.groupby()`
3. Aggregation with `.agg()`
4. Custom aggregation functions
5. Group filtering with `.filter()`

---

# Pandas II

We build on Pandas I with two new tools:

- **String `.str` methods** for working with text
- **Grouping** for summarizing subsets of a dataset

---

# Setup

We will use the **New York Baby Names** dataset for most examples. 

Baby names from New York from 1910–2018.

The dataset has five columns: `State`, `Sex`, `Year`, `Name`, and `Count`.

In [ ]:
# TODO: Import Pandas using its standard alias, pd



## Loading the Baby Names dataset

The Social Security Administration distributes state-level baby-name data in a ZIP file. The setup below downloads the file if necessary and reads the New York data.

This is setup code; you do **not** need to memorize it.

In [ ]:
# Import the tool needed to work with ZIP archives.
import zipfile

# Column names for the baby names dataset.
field_names = ["State", "Sex", "Year", "Name", "Count"]

# Read the New York baby names dataset.
baby_names = pd.read_csv(
    "namesbystate/NY.TXT",
    header=None,
    names=field_names
)

# Keep the time period used in the lecture slides (through 2018).
baby_names = baby_names[baby_names["Year"] <= 2018]

# Display the first 5 rows.
baby_names.head()

---

<div style="display: flex; align-items: center; gap: 50px;">

<div style="width: 40%; font-family: inherit;">

<h2>Baby Names DataFrame</h2>

<p>
Before working with strings, connect this dataset to Pandas I: each column is a <strong>Series</strong>, and all Series share the DataFrame's <strong>Index</strong>.
</p>

<p>
The <code>Name</code> Series contains text, so it is the column we will use to introduce Pandas string methods.
</p>

</div>

<div style="width: 60%; text-align: center;">

<img src="images/babynames_series.png"
     alt="Baby Names DataFrame anatomy"
     style="width: 100%; max-width: 900px;">

</div>

</div>

---

# String `.str` Methods

Pandas provides a collection of methods for working with text.

The `.str` accessor gives a Series access to **vectorized string operations**. Instead of writing a Python loop that processes one name at a time, we can ask Pandas to apply a string operation to every value in the Series.

The general pattern is:

```python
series.str.string_method()
```

## Example: names that start with `J`

Suppose we want to keep only names that start with the letter `J`.

From Pandas I, we know that filtering requires a Boolean condition: one `True` or `False` value for each row.

In [ ]:
# First, look at the Name Series.

baby_names["Name"].head(10)

One possible approach is to build the Boolean values ourselves with Python. This works, but it focuses on individual values rather than expressing the operation directly in Pandas.

In [ ]:
[name[0] == "J" for name in baby_names["Name"]]

In [ ]:
# Python approach: create one Boolean value for every name.

starts_with_j = [name[0] == "J" for name in baby_names["Name"]]
starts_with_j[10:15]

In [ ]:
# Use the Boolean values to filter the DataFrame.

baby_names[starts_with_j].head()

## The Pandas approach: `.str.startswith()`

Pandas gives us a more direct way to express the same idea.

`.str.startswith("J")` asks, for **every value in the Series**:

> Does this string start with `J`?

The result is a Boolean Series.

In [ ]:
# TODO: Check which names start with the letter J.
# .str.startswith() returns True or False for each value.



In [ ]:
# TODO: Use the Boolean Series to keep only names that start with J.



This is generally preferable because the code directly communicates our intention: **look at the Name Series and test whether each string starts with J**.

## Other useful `.str` methods

The slides introduce several common string tools:

- `.str.contains()` — does the string contain a pattern?
- `.str.lower()` — convert text to lowercase
- `.str.upper()` — convert text to uppercase
- `.str.capitalize()` — capitalize text
- `.str.count()` — count occurrences of a pattern
- `.str.isdigit()` — check whether the string contains only digits
- `.str.replace()` — replace text

<p>
<strong>More string methods:</strong>
Pandas provides many additional <code>.str</code> methods. You do not need to memorize them—use the documentation as a reference when needed.
</p>

<p>
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.html" target="_blank">
<strong>View the Pandas String Methods Reference →</strong>
</a>
</p>

In [ ]:
# TODO: Find names that contain the text "ice".



In [ ]:
# TODO: Convert the first few names to lowercase.



In [ ]:
# TODO: Convert the first few names to uppercase.



In [ ]:
# TODO: Replace the letter "a" with "_" in the first few names.
# regex=False means we are replacing the literal character "a",
# not using a regular-expression pattern.


# A regular expression (regex) is a pattern used to search text. 
# We will not use regex patterns here; `regex=False` tells Pandas to perform a simple text replacement.



### Useful extension from the previous notebook: string length

Another useful string operation is `.str.len()`, which returns the number of characters in each string.

In [ ]:
# TODO: Calculate the length of each name and display the first 5 results.



### Quick check

Return the first five rows whose `Name` ends with `"ert"`.

In [ ]:
# TODO: Filter to names that end with "ert".



---

# Grouping

Many data-analysis questions ask for a summary **for each group** rather than for the entire dataset.

Examples:

- total births **for each year**,
- maximum vote percentage **for each political party**,
- change in popularity **for each name**.

Pandas handles this pattern with `.groupby()`.

## The central idea: Split → Apply → Combine

A `.groupby()` operation can be understood in three stages:

1. **Split** — divide the DataFrame into groups according to one or more keys.
2. **Apply** — perform an operation on each group.
3. **Combine** — assemble the results into a new Pandas object.

This mental model is more important than memorizing individual `.groupby()` expressions.

<div style="display: flex; align-items: center; gap: 50px;">

<div style="width: 40%; font-family: inherit;">

<h3>Split</h3>

<p>
The diagram shows the original DataFrame being separated according to <code>Year</code>. Rows with the same year belong to the same group.
</p>

<p>
Calling <code>.groupby()</code> creates a <code>DataFrameGroupBy</code> object that represents these groups; it does not yet compute the final summary.
</p>

</div>

<div style="width: 60%; text-align: center;">

<img src="images/grouping_overview.png"
     alt="Split Apply Combine groupby diagram"
     style="width: 100%; max-width: 900px;">

</div>

</div>

In [ ]:
# TODO: Group the Baby Names data by Year.



The result is a `DataFrameGroupBy` object. Think of it as an object that knows **how the rows are divided into groups** and is waiting for us to specify what calculation to perform.

<div style="display: flex; align-items: center; gap: 50px;">

<div style="width: 40%; font-family: inherit;">

<h3>Apply and Combine</h3>

<p>
After splitting, we <strong>apply</strong> a calculation to each group. Pandas then <strong>combines</strong> the group-level results into a new object.
</p>

<p>
In the visual, values are grouped by <code>Year</code> and then summed, producing <strong>one result per year</strong>.
</p>

</div>

<div style="width: 60%; text-align: center;">

<img src="images/groupby_sum.png"
     alt="Groupby aggregation diagram"
     style="width: 100%; max-width: 900px;">

</div>

</div>

In [ ]:
# TODO: Find the total Count for each year.
# First group by Year, then select Count, then sum within each group.



Read the expression from left to right:

```python
baby_names.groupby("Year")["Count"].sum()
```

- `groupby("Year")` → split rows by year
- `["Count"]` → choose the values we want to summarize
- `.sum()` → add the counts inside each group
- Pandas combines the results → one total for each year

## Looking inside the groups

For learning purposes, we can iterate over a GroupBy object. Each iteration gives us the **group name** and the corresponding mini-DataFrame.

You usually will not need to write a loop like this in normal Pandas analysis; it is useful here because it makes the split step visible.

In [ ]:
# Display the first three groups only.

for i, (name, group) in enumerate(baby_names.groupby("Name")):
    print("Group:", name)
    display(group.head())
    if i == 2:
        break

---

# Aggregation

An **aggregation** reduces many values in a group to a smaller summary, often a single value.

Common aggregations include `sum`, `min`, `max`, `mean`, and `count`.

In [ ]:
# TODO: Find the total number of babies recorded for each name.



## `.agg()`

`.agg()` provides a general way to specify an aggregation function. For example, these two patterns express the same basic operation:

```python
grouped.max()
grouped.agg("max")
```

`.agg()` becomes especially useful when we want custom functions or more flexible summaries.

## Be careful: columns are aggregated independently

If we group by `Party` and take the maximum of several columns, Pandas calculates the maximum **separately for each column**.

The resulting values do **not necessarily come from the same original row**.

In [ ]:
# TODO: Load the elections dataset used for the grouping example.



In [ ]:
# TODO: Group by Party and calculate the maximum of each column.



For the Democratic group, Pandas independently finds:

- the maximum `Year`
- the alphabetically largest `Candidate`
- the maximum `Popular vote`
- the maximum `Result` (alphabetically)
- the maximum `%`

These values may come from **different election records**.

Therefore, a row in the aggregated result can look like a real observation even though no such row existed in the original DataFrame.

> **Key lesson:** An aggregated row contains **group-level summaries**. Do not automatically interpret it as an original observation.

---

# Custom aggregation functions

We are not limited to built-in functions. 

**Which baby name has experienced the greatest change in popularity?**

For this lecture, change is measured using the absolute max/min difference:


**AMMD = maximum Count − minimum Count**

For each name, we want to compare its largest and smallest recorded `Count`.

In [ ]:
# TODO: Define a function that returns max(series) - min(series).



Before applying the function to every name, test it on one name. This is a useful habit when building custom functions.

In [ ]:
# TODO: Select the Count values for Jennifer.



In [ ]:
# TODO: Calculate Jennifer's AMMD.



Now we want to repeat that calculation **for every name**. This is exactly the kind of problem `.groupby()` is designed to solve:

1. split the data by `Name`,
2. select `Count`,
3. apply `ammd` to each group,
4. combine the results.

In [ ]:
# TODO: Calculate AMMD for the Count values of every name.



Because `Name` is the grouping key, it becomes the Index of the aggregated result. The `Count` column now contains AMMD values rather than raw birth counts, so renaming it makes the result clearer.

In [ ]:
# TODO: Rename the aggregated Count column to AMMD.



In [ ]:
# TODO: Find the names with the largest change in popularity.



### Why selecting the correct column matters

If we apply `ammd` to every numeric column, Pandas will also calculate a range for `Year`. That calculation is valid mathematically, but it answers a different question.

Selecting `[["Count"]]` before `.agg(ammd)` makes our intention explicit: **measure change in popularity using Count**.

---

# Group filtering with `.filter()`

Boolean filtering from Pandas I asks whether **each row** satisfies a condition.

Group filtering asks a different question:

> Does this **entire group** satisfy the condition?

`GroupBy.filter()` keeps the original rows belonging to groups for which the supplied function returns `True`.

<div style="display: flex; align-items: center; gap: 50px;">

<div style="width: 40%; font-family: inherit;">

<h2>Filtering Happens per Group, Not per Row</h2>

<p>
The diagram shows the important distinction: the condition is evaluated on each <strong>grouped mini-DataFrame</strong>.
</p>

<p>
If a group passes the condition, the rows belonging to that group are retained.
</p>

</div>

<div style="width: 60%; text-align: center;">

<img src="images/groupby_filter.png"
     alt="Group filtering diagram"
     style="width: 100%; max-width: 900px;">

</div>

</div>

In [ ]:
# TODO: Keep only election-year groups whose maximum % is below 45.
# The lambda receives one grouped DataFrame at a time.



Compare the two ideas:

```python
# Row-level filtering
elections[elections["%"] < 45]

# Group-level filtering
elections.groupby("Year").filter(lambda group: group["%"].max() < 45)
```

The first can keep some rows from a year. The second either keeps **all rows from a year-group** or removes the entire group.

In [ ]:
# TODO: Group the data by Year.


# Look inside one group.
year_groups.get_group(1860)



---

# Useful GroupBy extensions

These are useful, but secondary to the core **split → apply → combine** idea.

## Group size

`.size()` tells us how many rows belong to each group.

In [ ]:
# TODO: Count how many election records belong to each party.



## Grouping by more than one column

We can create groups using a combination of column values. For example, grouping baby names by both `Year` and `Sex` creates a separate group for each year/sex combination.

In [ ]:
# TODO: Find total births for each Year and Sex combination.



The resulting Index has two levels (`Year` and `Sex`). We will encounter this kind of grouped output again later in the course; for now, focus on the idea that multiple grouping keys define more specific groups.

---

## Extra: Pivot Tables

A **pivot table** is another way to summarize data by groups.

Suppose we want to find the **total number of babies for each sex in each year**. We can already do this using `groupby()`:

In [ ]:
# TODO: Find the total Count for each Year and Sex combination.



The result is correct, but `Year` and `Sex` both appear as part of the Index.

A **pivot table** can arrange the same information in a more table-like format, with one grouping variable as the rows and another as the columns.

In [ ]:
# TODO: Create a pivot table showing total births by Year and Sex.
# index= determines the rows.
# columns= determines the columns.
# values= determines the values being summarized.
# aggfunc= determines how those values are combined.



### `groupby()` vs. `pivot_table()`

Both approaches perform **grouped aggregation**.

- `groupby()` is a general-purpose tool for splitting data into groups and applying operations.
- `pivot_table()` is especially useful when we want to arrange grouped results into a **two-dimensional summary table**.

---

# Quick Challenge

Using the Baby Names dataset, find the **10 names with the largest total `Count`** across all years in the dataset.

Your solution should:

1. group by `Name`,
2. select `Count`,
3. calculate the total for each group,
4. sort from largest to smallest, and
5. display the first 10 results.

In [ ]:
# Write your solution here


<details>
<summary><strong>How to read the solution</strong></summary>

```python
baby_names.groupby("Name")["Count"].sum().sort_values(ascending=False).head(10)
```

- `.groupby("Name")` splits rows by name.
- `["Count"]` selects the values to summarize.
- `.sum()` totals the counts within each name-group.
- `.sort_values(ascending=False)` places the largest totals first.
- `.head(10)` keeps the first 10 results.

</details>

---

# Before we finish

At this point, you should be able to:

- use the `Series.str.method()` pattern for text data,
- create Boolean filters with string methods such as `.str.startswith()` and `.str.contains()`,
- explain `.groupby()` using **split → apply → combine**,
- aggregate groups using methods such as `.sum()` and `.agg()`,
- apply a custom aggregation function, and
- explain why `GroupBy.filter()` filters **groups**, not individual rows.

You do not need to memorize every method. Use the examples in this notebook as a reference and focus on understanding what each stage of the Pandas expression is doing.